# Hard-negative fine-tune from P50 (v1)

14,747 positives (25%) + **2,000 hard negatives** (3,416 lines after tier-1 x2 oversampling) = 18.8% negatives.
Categories are driven by the measured FP taxonomy: dust/plaster 882, camera artifacts 563, neon/red lights 423,
orange PPE 288, sunset 209, person 199, chandelier 179, clouds 137, reflections 119, vehicle lights 82,
exhaust 75, fog 69, cave 62, steam 47, screen 19.

Starts from P50 with **low LR (1e-3)** (the earlier B run used the default LR and degraded), 15 epochs, 640.


In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available()
print("gpus", torch.cuda.device_count())


In [ ]:
import importlib.metadata as md, os
os.environ["YOLO_AUTOINSTALL"]="False"; os.environ["YOLO_VERBOSE"]="False"; os.environ["YOLO_OFFLINE"]="True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"
def ver(n):
    try: return md.version(n)
    except Exception: return None
if ver("ultralytics") != "8.4.147":
    !pip install -q --no-deps --progress-bar off ultralytics==8.4.147 ultralytics-thop
print({"ultralytics": ver("ultralytics"), "torch": ver("torch")}, flush=True)


In [ ]:
import json, os
from pathlib import Path

DATA = str(sorted(Path("/kaggle/input").rglob("manifest_summary.json"))[0].parent)
hits = sorted(Path("/kaggle/input").rglob("pilot_train.py"))
pref = [h for h in hits if "firesight-pilot-code" in str(h)] or hits
CODE = str(pref[0].parent)
assert "def resolve_list" in (Path(CODE)/"pilot_train.py").read_text()
P50 = sorted(Path("/kaggle/input").rglob("P50/weights/best.pt"))[0]
NEG = sorted(Path("/kaggle/input").rglob("negatives.jsonl"))[0]
print("DATA",DATA); print("CODE",CODE); print("P50",P50); print("NEG",NEG, flush=True)

pkg = Path(DATA)
def resolve(line):
    p = Path(line.strip())
    return str(p if p.is_absolute() else (pkg/p).resolve())

train25 = [resolve(l) for l in open(pkg/"lists"/"train_25.txt") if l.strip()]
val = [resolve(l) for l in open(pkg/"lists"/"val_early.txt") if l.strip()]
negs = [json.loads(l) for l in open(NEG) if l.strip()]
neg_lines = [str((pkg/"images"/n["package_file"]).resolve()) for n in negs for _ in range(int(n.get("repeat",1)))]
missing = [p for p in neg_lines if not Path(p).exists()]
assert not missing, f"{len(missing)} negative paths missing, e.g. {missing[:3]}"

out = Path("/kaggle/working/ft"); (out/"lists").mkdir(parents=True, exist_ok=True)
lines = train25 + neg_lines
(out/"lists"/"train.txt").write_text("\n".join(lines)+"\n", encoding="utf-8")
(out/"lists"/"val.txt").write_text("\n".join(val)+"\n", encoding="utf-8")
(out/"data.yaml").write_text("path: /kaggle/working/ft\ntrain: /kaggle/working/ft/lists/train.txt\n"
                             "val: /kaggle/working/ft/lists/val.txt\nnames:\n  0: fire\n  1: smoke\nnc: 2\n",
                             encoding="utf-8")
print("positives", len(train25), "negative lines", len(neg_lines), "neg share", round(len(neg_lines)/len(lines),4), flush=True)


In [ ]:
import os, subprocess
OUT = "/kaggle/working/runs/FT_HN"
cmd = ["timeout","10800","python","-u",f"{CODE}/pilot_train.py","--dataset",DATA,"--fraction","25",
       "--data-yaml","/kaggle/working/ft/data.yaml","--n-train",str(len(lines)),
       "--model",str(P50),"--epochs","15","--imgsz","640","--batch","64","--device","0",
       "--workers","2","--amp","--lr0","0.001","--patience","5","--out",OUT]
print("RUN:", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True, stderr=subprocess.STDOUT, env={**os.environ, "PYTHONUNBUFFERED":"1"})


In [ ]:
import gc, json, subprocess
import torch
from ultralytics import YOLO
from pathlib import Path

list_file = "/kaggle/working/val_paths.txt"
open(list_file,"w").write("\n".join([str(Path(DATA)/l.strip()) if not Path(l.strip()).is_absolute() else l.strip()
                                      for l in open(Path(DATA)/"lists/val_cal.txt")] +
                                     [str(Path(DATA)/l.strip()) if not Path(l.strip()).is_absolute() else l.strip()
                                      for l in open(Path(DATA)/"lists/val_report.txt")]) + "\n")
model = YOLO(f"{OUT}/weights/best.pt")
rows=[]
for res in model.predict(list_file, imgsz=640, conf=0.01, batch=8, half=True, verbose=False, stream=True):
    dets=[{"label":"fire" if int(b.cls.item())==0 else "smoke","confidence":round(float(b.conf.item()),6),
           "box":[float(v) for v in b.xyxy[0].tolist()]} for b in res.boxes]
    rows.append({"id":Path(res.path).stem,"detections":dets})
preds="/kaggle/working/preds_FT_HN.jsonl"
open(preds,"w").write("\n".join(json.dumps(r) for r in rows)+"\n")
del model; gc.collect(); torch.cuda.empty_cache()
for t in ("0.015","0.02"):
    subprocess.run(["python",f"{CODE}/pilot_eval.py","--dataset",DATA,"--preds",preds,"--model","FT_HN",
                    "--target-fpr",t,"--out",f"/kaggle/working/metrics_FT_HN_{t}.json"], check=True)


In [ ]:
import json
for t in ("0.015","0.02"):
    m=json.load(open(f"/kaggle/working/metrics_FT_HN_{t}.json")); c=m["splits"]["val"]["combined"]; f=m["splits"]["val"]["fpr_per_negative_image"]
    print(t,"thr",m["thresholds"],"acc",c["accuracy"],"recall",c["recall"],"ci",c["recall_95ci"],"fpr",f["fpr"],
          "map50",m["splits"]["val"]["box_ap"]["fire"]["ap50"],"/",m["splits"]["val"]["box_ap"]["smoke"]["ap50"])
print(open(f"{OUT}/run_summary.json").read())
